# DFT Ion Forces

This notebook checks force consistency for the local ion-pseudopotential path. We compare analytic local forces with fixed-density finite differences, then run a compact SCF total-energy force check.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import DFTSystem, DiracExchange, Ion, IonCollection, LocalPseudopotentialField, RealSpaceGrid, SCFConfig, read_gth, scf_total_energy_forces

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

The fixed-density test isolates the local potential derivative. This is the cleanest check before SCF response effects enter.

In [ ]:
gth = read_gth(ROOT / "vendors/quantum-espresso/pseudo/H-q1.gth", element="H")
grid = RealSpaceGrid((6, 6, 6), (6.0, 6.0, 6.0))
coords = np.array(grid.coordinates())
density = np.exp(-np.sum((coords - np.array([3.4, 3.0, 3.0])) ** 2, axis=-1) / 0.8).astype(np.float32)
density *= 1.0 / (np.sum(density) * grid.dv)

def local_energy(center_x):
    field = LocalPseudopotentialField(IonCollection([Ion("H", (center_x, 3.0, 3.0), gth)]))
    return float(np.sum(density * np.array(field.field(grid))) * grid.dv)

field = LocalPseudopotentialField(IonCollection([Ion("H", (3.0, 3.0, 3.0), gth)]))
analytic = np.array(field.forces(density, grid))[0, 0]
epsilon = 1e-3
finite_difference = -(local_energy(3.0 + epsilon) - local_energy(3.0 - epsilon)) / (2.0 * epsilon)
{"analytic_force_x": analytic, "finite_difference_force_x": finite_difference, "abs_error": abs(analytic - finite_difference)}

In [ ]:
xs = np.linspace(2.8, 3.2, 9)
energies = [local_energy(x) for x in xs]
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.plot(xs, energies, marker="o")
ax.axvline(3.0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("ion x / bohr")
ax.set_ylabel("fixed-density local energy")
ax.set_title("local ion force finite-difference surface");

The SCF force check reruns the electronic problem after displacing the ion. This is stricter than the fixed-density check, but it is still a prototype validation for a local-only ion model.

In [ ]:
system = DFTSystem(
    cell=(6.0, 6.0, 6.0),
    grid_shape=(4, 4, 4),
    ions=IonCollection([Ion("H", (3.0, 3.0, 3.0), gth)]),
)
check = scf_total_energy_forces(
    system,
    config=SCFConfig(max_iterations=2, solver="dense", seed=9),
    xc_functional=DiracExchange(),
    displacement=1e-3,
)
check.to_dict()